In [99]:
import re
from collections import defaultdict, Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem import PorterStemmer

#parametres d'affichage des tableaux
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [100]:
documents = {
    "D1": "Information retrieval helps users find relevant documents.",
    "D2": "Information Retrieval systems index documents and documents.",
    "D3": "Search engines retrieve relevant web documents.",
    "D4": "Document indexing supports efficient information retrieval.",
    "D5": "Machine learning improves search engine ranking.",
    "D6": "Search systems use indexed documents to retrieve information.",
    "D7": "The search engine retrieves documents; the documents are searchable.",
    "D8": "Indexing documents makes information retrieval more efficient.",
    "D9": "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections."
}

#PARTIE 1 : Text preprocessing

# etape B : fonction qui fait le preprocessing de text (suprimmer les majuscules, les espaces, les vergules.. puis tokenisation)
def preprocessing(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    return tokens

#test de la fonction
text = "DOCUMENTS contain information about document indexing."
result = preprocessing(text)
print(result)

#applying the function for all the documents
collectionA = {doc_id: preprocessing(text) for doc_id, text in documents.items()}
print(collectionA)

# etape C : supression des stops words
stop_words = set(stopwords.words('english'))
print(stop_words)  #afficher la liste des stops words

def suprimer_stopwords(tokens):
  tokens_without_stopwords = [w for w in tokens if w not in stop_words]
  return tokens_without_stopwords

#test de la fonction
text = "DOCUMENTS contain for more information about document indexing."
result = preprocessing(text)
result = suprimer_stopwords(result)
print(result)

#appliquer la fonction pour tous les documents
collectionB = {doc_id: suprimer_stopwords(tokens) for doc_id, tokens in collectionA.items()}
print(collectionB)

# etape D : stemming

stemmer = PorterStemmer()
def stemming(tokens):
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

collectionC = {doc_id: stemming(tokens) for doc_id, tokens in collectionB.items()}
print(collectionC)

['documents', 'contain', 'information', 'about', 'document', 'indexing']
{'D1': ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents'], 'D2': ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents'], 'D3': ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents'], 'D4': ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval'], 'D5': ['machine', 'learning', 'improves', 'search', 'engine', 'ranking'], 'D6': ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information'], 'D7': ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable'], 'D8': ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient'], 'D9': ['documents', 'contain', 'information', 'about', 'document', 'indexing'], 'D10': ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']}
{"i've", 'm', 'for', 'more', 'we', 'be', 're', "wouldn't", '

In [101]:
#PARTIE 2 : Vocabulary and document representation

def construire_vocabulaire(collection):
    vocabulaire = set() #conserver uniquiment les mots uniques
    for tokens in collection.values():
        vocabulaire.update(tokens)
    return sorted(vocabulaire) #trier les mots

# Building the vocabulary for versions A, B and C :

VocabulaireA = construire_vocabulaire(collectionA)
VocabulaireB = construire_vocabulaire(collectionB)
VocabulaireC = construire_vocabulaire(collectionC)

print("\nVocabulaire A : ", VocabulaireA)
print("Vocabulaire B : ", VocabulaireB)
print("Vocabulaire C : ", VocabulaireC)


# Implementing the binary term–document matrix :

def build_term_document_matrix(collection):
    vocabulaire = construire_vocabulaire(collection)
    matrix = {}  #initialiser la matrice avec des 0

    for doc_id in collection:
        matrix[doc_id] = {}
        for term in vocabulaire:
            matrix[doc_id][term] = 0

    for doc_id, tokens in collection.items():  #remplir la matrice binaire
        for term in tokens:
            matrix[doc_id][term] = 1

    return matrix

#tester la fonction
binary_matrixA = build_term_document_matrix(collectionA)
binary_matrixB = build_term_document_matrix(collectionB)
binary_matrixC = build_term_document_matrix(collectionC)

print("\nBinary matrix A\n")
print(pd.DataFrame(binary_matrixA).T)
print("\n")
print("Binary matrix B\n")
print(pd.DataFrame(binary_matrixB).T)
print("\n")
print("Binary matrix C\n")
print(pd.DataFrame(binary_matrixC).T)

# Building the term frequency matrix

def build_term_frequency_matrix(collection):
    vocabulaire = construire_vocabulaire(collection)
    matrix = {}  #initialiser la matrice

    for doc_id in collection:
        matrix[doc_id] = {}
        for term in vocabulaire:
            matrix[doc_id][term] = 0

    for doc_id, tokens in collection.items():  #remplir la matrice
        term_freq = Counter(tokens)            # compter la frequence d'un element dans une liste
        for term, freq in term_freq.items():
            matrix[doc_id][term] = freq

    return matrix

#tester la fonction
termFreq_matrixA = build_term_frequency_matrix(collectionA)
termFreq_matrixB = build_term_frequency_matrix(collectionB)
termFreq_matrixC = build_term_frequency_matrix(collectionC)

print("\nTerm frequency matrix A\n")
print(pd.DataFrame(termFreq_matrixA).T)
print("\nTerm frequency matrix B\n")
print(pd.DataFrame(termFreq_matrixB).T)
print("\nTerm frequency matrix C\n")
print(pd.DataFrame(termFreq_matrixC).T)

# La matrice binaire indique la presence d'un terme (1) ou bien son absence (0) dans un document, mais la matrice de fréquence compte le nombre d'occurances de chaque mot prenant aussi en compte les repititions


Vocabulaire A :  ['about', 'and', 'are', 'collections', 'contain', 'document', 'documents', 'efficient', 'engine', 'engines', 'find', 'helps', 'improves', 'index', 'indexed', 'indexing', 'information', 'learning', 'machine', 'makes', 'more', 'organise', 'ranking', 'relevant', 'retrieval', 'retrieve', 'retrieves', 'search', 'searchable', 'supports', 'systems', 'the', 'to', 'use', 'users', 'web']
Vocabulaire B :  ['collections', 'contain', 'document', 'documents', 'efficient', 'engine', 'engines', 'find', 'helps', 'improves', 'index', 'indexed', 'indexing', 'information', 'learning', 'machine', 'makes', 'organise', 'ranking', 'relevant', 'retrieval', 'retrieve', 'retrieves', 'search', 'searchable', 'supports', 'systems', 'use', 'users', 'web']
Vocabulaire C :  ['collect', 'contain', 'document', 'effici', 'engin', 'find', 'help', 'improv', 'index', 'inform', 'learn', 'machin', 'make', 'organis', 'rank', 'relev', 'retriev', 'search', 'searchabl', 'support', 'system', 'use', 'user', 'web']

In [102]:
# PARTIE 3 : Inverted index with positions

# Constructing a small index for the terms information, retrieval and documents (used collection A)
small_index = {
    'information': {
        'D1': [0],
        'D2': [0]
    },
    'retrieval': {
        'D1': [1],
        'D2': [1]
    },
    'documents': {
        'D1': [6],
        'D2': [4,5],
        'D3': [5]
    }
}

#Implementing the index
def build_inverted_index(collection):
    inverted_index = defaultdict(lambda: defaultdict(list))

    for doc_id, tokens in collection.items():
        for position, term in enumerate(tokens):
            inverted_index[term][doc_id].append(position)

    #conversion en dict classiques pour l'affichage
    return {
        term: dict(doc_positions)
        for term, doc_positions in inverted_index.items()
    }

def search_term(inverted_index, term):
    if term not in inverted_index:
        return []
    return sorted(inverted_index[term].keys())

#tester la fonction
Index_A = build_inverted_index(collectionA)
index_B = build_inverted_index(collectionB)
index_C = build_inverted_index(collectionC)

#exemple collection A
for term in ["information", "retrieval", "document"]:
    print(term, "->", Index_A.get(term))

information -> {'D1': [0], 'D2': [0], 'D4': [4], 'D6': [7], 'D8': [3], 'D9': [2]}
retrieval -> {'D1': [1], 'D2': [1], 'D4': [5], 'D8': [4], 'D10': [0]}
document -> {'D4': [0], 'D9': [4], 'D10': [5]}


In [103]:
# PARTIE 4 : Effect of preprocessing

# Les différents mesures
def compter_mesures(collection,inverted_index):
  vocab_size = len(construire_vocabulaire(collection))
  total_tokens = sum(len(tokens) for tokens in collection.values())
  avg_tokens_per_doc = total_tokens / len(collection)
  total_postings = sum(len(docs) for docs in inverted_index.values())
  total_positions = sum(
        len(positions)
        for docs in inverted_index.values()
        for positions in docs.values()
    )
  return {
        "Vocabulary size": vocab_size,
        "Avg tokens/doc": round(avg_tokens_per_doc, 2),
        "Total postings": total_postings,
        "Total stored positions": total_positions,
    }

#pour afficher les mesures sous forme de tableau pour les 3 collections A,B,C
mesures = pd.DataFrame({
    "Version A": compter_mesures(collectionA, Index_A),
    "Version B": compter_mesures(collectionB, index_B),
    "Version C": compter_mesures(collectionC, index_C)
}).T
print(mesures)

           Vocabulary size  Avg tokens/doc  Total postings  \
Version A             36.0             6.9            66.0   
Version B             30.0             6.1            59.0   
Version C             24.0             6.1            57.0   

           Total stored positions  
Version A                    69.0  
Version B                    61.0  
Version C                    61.0  


In [104]:
# PARTIE 5 : Vérification

def verifications():
    print("************************* Tests de search_term() ***************************")
    #term occurring in only one document (learn dans D5 de collectionC)
    res_seul = search_term(index_C, "learn")
    attendu_seul = sorted([doc for doc, toks in collectionC.items() if "learn" in toks])
    print("Terme unique (learn en C) :", res_seul, res_seul == attendu_seul)

    #term occurring in multiple documents (document dans collectionC)
    res_multi = search_term(index_C, "document")
    attendu_multi = sorted([doc for doc, toks in collectionC.items() if "document" in toks])
    print("Terme multiple (document en C) :", res_multi, res_multi == attendu_multi)

    #term whose form changes after stemming
    print("Avant stemming (B) 'documents' ->", search_term(index_B, "documents"))
    print("Apres stemming (C) 'documents' ->", search_term(index_C, "documents"))
    print("Apres stemming (C) 'document'  ->", search_term(index_C, "document"))

    #unknown term that does not occur in the collection
    res_inconnu = search_term(index_C, "Inconnu")
    print("Terme inconnu :", res_inconnu == [])

    print("\n********************* Vérification des positions dans l'index inversé *********************")
    positions_ok = True
    for nom, col, idx in [("A", collectionA, Index_A), ("B", collectionB, index_B), ("C", collectionC, index_C)]:
        for term, docs in idx.items():
            for doc_id, positions in docs.items():
                for pos in positions:
                    if col[doc_id][pos] != term:
                        print(f"Erreur position [{nom}] {term} dans {doc_id} à {pos}")
                        positions_ok = False
    print("Toutes les positions correspondent aux tokens :", positions_ok)

    print("\n********************* Comparaison small_index vs Index_A sur D1-D3 *********************")
    for term, docs in small_index.items():
        for doc_id, pos_manuel in docs.items():
            pos_auto = Index_A.get(term, {}).get(doc_id, [])
            identique = (pos_manuel == pos_auto)
            remarque = "" if identique else " (erreur dans small_index : 'and' est à l'index 5 dans D2)"
            print(f"{term} [{doc_id}] -> manuel: {pos_manuel} | auto: {pos_auto} | identique: {identique}{remarque}")

    print("\n********************* Vérification des matrices binaire et fréquences ****************************")
    matrices_ok = True
    for col, b_mat, tf_mat in [
        (collectionA, binary_matrixA, termFreq_matrixA),
        (collectionB, binary_matrixB, termFreq_matrixB),
        (collectionC, binary_matrixC, termFreq_matrixC),
    ]:
        for doc_id, row in b_mat.items():
            for term, val_b in row.items():
                freq_attendue = col[doc_id].count(term)
                val_tf = tf_mat[doc_id][term]
                b_attendu = 1 if freq_attendue > 0 else 0
                if val_b not in (0, 1) or val_b != b_attendu or val_tf != freq_attendue:
                    matrices_ok = False
    print("Toutes les valeurs des matrices binaire et TF sont valides :", matrices_ok)

verifications()

************************* Tests de search_term() ***************************
Terme unique (learn en C) : ['D5'] True
Terme multiple (document en C) : ['D1', 'D10', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9'] True
Avant stemming (B) 'documents' -> ['D1', 'D10', 'D2', 'D3', 'D6', 'D7', 'D8', 'D9']
Apres stemming (C) 'documents' -> []
Apres stemming (C) 'document'  -> ['D1', 'D10', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9']
Terme inconnu : True

********************* Vérification des positions dans l'index inversé *********************
Toutes les positions correspondent aux tokens : True

********************* Comparaison small_index vs Index_A sur D1-D3 *********************
information [D1] -> manuel: [0] | auto: [0] | identique: True
information [D2] -> manuel: [0] | auto: [0] | identique: True
retrieval [D1] -> manuel: [1] | auto: [1] | identique: True
retrieval [D2] -> manuel: [1] | auto: [1] | identique: True
documents [D1] -> manuel: [6] | auto: [6] | identique: True
documents [D2] ->

In [107]:
# PARTIE 6 : GitHub Deliverables

import json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
dossier_results = Path('/content/drive/MyDrive/IR-TP1/results')
dossier_results.mkdir(parents=True, exist_ok=True)
for suffix, col in [("A", collectionA), ("B", collectionB), ("C", collectionC)]:
    with open(dossier_results / f"tokens_{suffix}.json", "w", encoding="utf-8") as f:
        json.dump(col, f, indent=2, ensure_ascii=False)
for suffix, vocab in [("A", VocabulaireA), ("B", VocabulaireB), ("C", VocabulaireC)]:
    with open(dossier_results / f"vocabulary_{suffix}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab))
for suffix, b_mat, tf_mat in [("A", binary_matrixA, termFreq_matrixA), ("B", binary_matrixB, termFreq_matrixB), ("C", binary_matrixC, termFreq_matrixC)]:
    pd.DataFrame(b_mat).to_csv(dossier_results / f"binary_matrix_{suffix}.csv")
    pd.DataFrame(tf_mat).to_csv(dossier_results / f"tf_matrix_{suffix}.csv")
for suffix, idx in [("A", Index_A), ("B", index_B), ("C", index_C)]:
    with open(dossier_results / f"inverted_index_{suffix}.json", "w", encoding="utf-8") as f:
        json.dump(idx, f, indent=2, ensure_ascii=False)
mesures.to_csv(dossier_results / "preprocessing_comparison.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
